# Query expansion for financial page retrieval

**Research question:** Can alternative formulations of a question improve retrieval from financial reports, and does optimizing those formulations help beyond a manually written prompt?

The experiment extends the existing ViDoRe Finance benchmark. It reuses the same page corpus, page labels and Qwen3-VL page embeddings. The new component is the search question: the original wording, LangChain MultiQueryRetriever alternatives, or DSPy alternatives before and after prompt optimization. No answer reader or answer judge is evaluated.

All results below are loaded from saved, executed runs. Reopening this notebook does not make API calls or regenerate embeddings.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'results/manifest.json').exists():
    ROOT = ROOT / 'outputs/vidore_query_expansion'
RESULTS = ROOT / 'results'
load = lambda name: json.loads((RESULTS / name).read_text(encoding='utf-8'))
manifest = load('manifest.json')
audit = load('data_audit.json')
protocol = load('protocol.json')
selection = load('selection.json')
summary = pd.read_csv(RESULTS / 'summary.csv')
assert manifest['queries'] == 1854 and manifest['pages'] == 2942
display(pd.DataFrame(audit['split_counts']).T.rename(columns={'english':'English questions', 'all_variants':'All language variants'}))
print('Search pool:', manifest['pages'], 'pages from six financial reports')
print('Generator:', protocol['generator_model'])

,English questions,All language variants
train,31,186
validation,31,186
test,247,1482


Search pool: 2942 pages from six financial reports
Generator: gpt-4o-mini-2024-07-18


## 1. Data and evaluation split

There are **309 English questions**, each with five translated versions, for **1,854 query variants**. Every query searches the complete **2,942-page** English corpus. Translations are repeated information needs, not 1,854 independent English questions.

Questions with identical full relevance-label signatures stay together. This conservative grouping keeps translations together and also groups some distinct questions sharing the same labels. There are 31 English training questions, 31 validation questions and 247 optimizer-held-out test questions. Training examples support DSPy's instruction proposal; validation selects prompts and retrieval settings. The headline table uses the 247 English test questions.

These queries were evaluated in the previous benchmark, so the test set is held out from this optimizer rather than a newly unseen benchmark. Six shared reports also limit the breadth of generalization.

In [2]:
splits = pd.read_csv(RESULTS / 'splits.csv')
assert splits.groupby('group')['split'].nunique().max() == 1
display(splits.groupby(['split', 'language']).size().unstack().reindex(['train','validation','test']))
display(pd.Series({'Relevance-signature groups': audit['relevance_groups'],
                   'Groups crossing splits': 0,
                   'Pages available for every search': manifest['pages']}))

language,english,french,german,italian,portuguese,spanish
split,,,,,,
train,31,31,31,31,31,31
validation,31,31,31,31,31,31
test,247,247,247,247,247,247


Relevance-signature groups           299
Groups crossing splits                 0
Pages available for every search    2942
dtype: int64

## 2. Retrieval methods and query generators

| Search backend | What is indexed | Representation |
|---|---|---|
| BM25 | Full OCR/Markdown page text | Lexical term matching |
| Dense OCR | OCR/Markdown page text | Qwen3-VL-Embedding-2B vectors |
| Dense image | Page images | Qwen3-VL-Embedding-2B visual vectors |
| OCR/image fusion | Both dense rankings | Weighted reciprocal rank fusion |

The page vectors are reused from the pinned prior run: a 4,096-token text cap and 786,432-pixel image cap. Generated questions use the same query encoder and financial retrieval instruction as the original questions. No embedding model is fine-tuned. OCR/image fusion is dense multimodal fusion; it does not combine BM25 and dense scores.

**Original:** search once with the user's question. **MultiQuery:** use the actual LangChain `MultiQueryRetriever.generate_queries()` interface with general and financial prompt candidates. **DSPy initial:** a typed `question -> list[str]` program. **DSPy optimized:** MIPROv2 selects its instruction using retrieval nDCG, rather than generated-answer accuracy. This is a retrieval-specific DSPy experiment.

Each generator produces five alternatives. Development tuning chooses whether to use the first 1, 3 or 5 and whether to retain the original question. Candidate retrieval and fusion use one shared ranked-search implementation for all generators. LangChain's unordered document union is therefore replaced with an explicit, reproducible ranking:

\[
S(d)=\sum_{j=0}^{m}\frac{w_j\,\mathbf{1}[d\in L_j[:D]]}{c+\operatorname{rank}_j(d)}.
\]

Here, $m$ is the number of alternatives, $D$ is per-query candidate depth, $c$ is the RRF constant, and $w_0$ is the original-question weight. Alternative-query weights are 1. An original weight of 0 excludes the original. Each list uses stable page-ID tie-breaking. Dense OCR/image fusion happens within each query first, with its own fixed constant of 60 and depth of 100.

## 3. Development-only parameter tuning

The primary selection metric is **English validation nDCG@10**, with Recall@10 as the first tie-breaker. Further ties favor fewer alternatives, shallower candidate sets, retaining the original, and a constant near 60.

The grid covers BM25 term-frequency saturation and length normalization; OCR/image fusion weight; general versus financial prompts; number of query alternatives; candidate depth; RRF constant; and original-query weight. Five alternatives are generated once per prompt/question, and prefix subsets are reused across configurations, so the grid does not require an API call for every setting.

DSPy uses two optimizer seeds, six instruction candidates, 12 search trials per seed plus the default-program evaluation, and zero demonstrations in the final programs. Its optimization uses BM25 with three alternatives, original weight 1, depth 100 and RRF constant 60. The selected instruction then transfers to the other backends; dense/image retrieval is not used inside MIPROv2's scoring loop.

In [3]:
grid = {
    'BM25 k1': protocol['bm25_k1'], 'BM25 b': protocol['bm25_b'],
    'OCR weight in OCR/image fusion': protocol['ocr_fusion_weights'],
    'Alternatives used': protocol['expansion_counts'],
    'Candidates per query': protocol['candidate_depths'],
    'Query-fusion RRF constant': protocol['rrf_constants'],
    'Original-question weight': protocol['original_query_weights'],
}
display(pd.DataFrame([{'Parameter': name, 'Values tested': str(values)} for name, values in grid.items()]))
print('Selected common baseline settings:', selection['baseline'])
tuning = pd.read_csv(RESULTS / 'expansion_tuning.csv')
print('Expansion configurations evaluated on validation:', len(tuning))
selected_rows = [dict(Backend=backend, Family=family, **params)
                 for backend, methods in selection['expansion'].items()
                 for family, params in methods.items()]
display(pd.DataFrame(selected_rows)[['Backend','Family','strategy','count','depth','rrf_constant','original_weight','ndcg_at_10']].round(4))
display(pd.concat([pd.read_json(RESULTS / f'optimizer_trials_{seed}.json').assign(seed=seed)
                   for seed in protocol['dspy_optimizer']['seeds']], ignore_index=True).groupby('seed')['ndcg_at_10'].agg(['count','min','max']).round(4))

,Parameter,Values tested
0,BM25 k1,"[0.8, 1.2, 1.5, 2.0]"
1,BM25 b,"[0.0, 0.5, 0.75, 1.0]"
2,OCR weight in OCR/image fusion,"[0.0, 0.25, 0.5, 0.75, 1.0]"
3,Alternatives used,"[1, 3, 5]"
4,Candidates per query,"[20, 50, 100]"
5,Query-fusion RRF constant,"[20, 60, 100]"
6,Original-question weight,"[0.0, 0.5, 1.0, 2.0]"


Selected common baseline settings: {'k1': 0.8, 'b': 0.75, 'ocr_weight': 0.25}
Expansion configurations evaluated on validation: 2160


,Backend,Family,strategy,count,depth,rrf_constant,original_weight,ndcg_at_10
0,bm25,multiquery,mq_financial,1,20,20,2.0,0.6544
1,bm25,dspy_initial,dspy_initial,1,50,20,2.0,0.6516
2,bm25,dspy_optimized,dspy_seed_19,3,50,20,0.0,0.6639
3,ocr,multiquery,mq_general,1,20,60,2.0,0.6225
4,ocr,dspy_initial,dspy_initial,1,20,60,2.0,0.6132
5,ocr,dspy_optimized,dspy_seed_19,5,100,20,2.0,0.6310
6,image,multiquery,mq_general,5,50,60,2.0,0.7061
7,image,dspy_initial,dspy_initial,1,50,20,0.5,0.7120
8,image,dspy_optimized,dspy_seed_19,3,50,20,0.5,0.7232
9,combined,multiquery,mq_general,1,20,100,0.5,0.7312


,count,min,max
seed,,,
7,13,0.5900,0.6191
19,13,0.5968,0.6334


## 4. What the metrics mean

All methods use the same published page labels and report the same metrics at **1, 5, 10 and 20**. These cutoffs control the number of final returned pages. They differ from candidate depth, which is the number of pages contributed by each query before fusion.

* **Hit@k:** at least one judged relevant page appears in the top k.
* **Recall@k / judged-page coverage:** the fraction of judged relevant pages retrieved. It does not measure the fraction of required answer facts.
* **Precision@k:** relevant pages retrieved divided by k.
* **MRR@k:** reciprocal rank of the first relevant page, with zero if absent.
* **nDCG@k:** ranking quality using the dataset's linear grades 1 and 2, discounted by rank and normalized by the ideal ordering.
* **All-judged-page hit@k:** every judged relevant page was returned. Some pages can be alternative sources, so this can be stricter than answer sufficiency.
* **Complete-answer-page hit@k:** at least one grade-2 page was returned, evaluated only for questions with a grade-2 label. This uses the author's page judgment, rather than newly validated span/fact annotations.

No reader is run, so these are retrieval results rather than answer correctness or groundedness scores. Unjudged pages are treated as nonrelevant for scoring; the annotations may not list every usable page.

In [4]:
test = summary[(summary.scope == 'split_language') & (summary['group'] == 'test/english')].copy()
headline = test[test.k == 10].set_index('method')[['queries','hit','recall','precision','mrr','ndcg','all_judged_hit','complete_page_hit','complete_page_eligible_queries']]
headline.insert(1, 'hit_at_5', test[test.k == 5].set_index('method')['hit'])
display(headline.sort_index().round(4))

,queries,hit_at_5,hit,recall,precision,mrr,ndcg,all_judged_hit,complete_page_hit,complete_page_eligible_queries
method,,,,,,,,,,
bm25/dspy_initial,247,0.8178,0.8826,0.5807,0.1862,0.6465,0.5243,0.3522,0.8167,180
bm25/dspy_optimized,247,0.8178,0.8907,0.5823,0.1911,0.6462,0.5188,0.3522,0.8278,180
bm25/multiquery,247,0.7895,0.8826,0.5877,0.1895,0.6336,0.5191,0.3644,0.8222,180
bm25/original,247,0.7814,0.8745,0.5644,0.1810,0.6247,0.5037,0.3401,0.8111,180
combined/dspy_initial,247,0.8623,0.9028,0.6275,0.2134,0.6999,0.5743,0.3765,0.8500,180
combined/dspy_optimized,247,0.8543,0.9150,0.6279,0.2134,0.6991,0.5712,0.3765,0.8556,180
combined/multiquery,247,0.8583,0.9069,0.6191,0.2077,0.6736,0.5540,0.3765,0.8500,180
combined/original,247,0.8462,0.9150,0.6361,0.2138,0.6979,0.5760,0.3846,0.8556,180
image/dspy_initial,247,0.8664,0.8947,0.6225,0.2081,0.7077,0.5750,0.3846,0.8500,180


## 5. Cutoff sensitivity and uncertainty

The first table compares matched cutoffs across all methods. Deeper retrieval can improve page coverage while returning more irrelevant pages. It does not automatically improve the final answer.

The second table gives paired differences from each backend's original-question baseline at k=10, with 2,000 bootstrap resamples of relevance-signature groups. These are pointwise descriptive intervals. They do not correct for multiple comparisons or establish performance on new reports.

In [5]:
display(test.pivot(index='method', columns='k', values='hit').sort_index().round(4).rename_axis(columns='Hit cutoff'))
display(test.pivot(index='method', columns='k', values='recall').sort_index().round(4).rename_axis(columns='Recall cutoff'))
paired = pd.read_csv(RESULTS / 'paired_comparisons.csv')
display(paired[(paired.k == 10) & (paired.metric == 'ndcg')][['backend','family','difference','paired_group_bootstrap_lower_95','paired_group_bootstrap_upper_95','wins','losses','ties']].round(4))

Hit cutoff,1,5,10,20
method,,,,
bm25/dspy_initial,0.5182,0.8178,0.8826,0.9393
bm25/dspy_optimized,0.5142,0.8178,0.8907,0.9474
bm25/multiquery,0.5061,0.7895,0.8826,0.9231
bm25/original,0.4980,0.7814,0.8745,0.9231
combined/dspy_initial,0.5830,0.8623,0.9028,0.9717
combined/dspy_optimized,0.5789,0.8543,0.9150,0.9636
combined/multiquery,0.5425,0.8583,0.9069,0.9352
combined/original,0.5749,0.8462,0.9150,0.9717
image/dspy_initial,0.5951,0.8664,0.8947,0.9717


Recall cutoff,1,5,10,20
method,,,,
bm25/dspy_initial,0.2318,0.4657,0.5807,0.6818
bm25/dspy_optimized,0.2148,0.4678,0.5823,0.6922
bm25/multiquery,0.2284,0.4601,0.5877,0.6643
bm25/original,0.2206,0.4514,0.5644,0.6643
combined/dspy_initial,0.2461,0.5162,0.6275,0.7484
combined/dspy_optimized,0.2397,0.5111,0.6279,0.7458
combined/multiquery,0.2145,0.5127,0.6191,0.7024
combined/original,0.2455,0.5109,0.6361,0.7484
image/dspy_initial,0.2520,0.5170,0.6225,0.7332


,backend,family,difference,paired_group_bootstrap_lower_95,paired_group_bootstrap_upper_95,wins,losses,ties
5,bm25,multiquery,0.0153,0.0007,0.0302,87,51,109
14,bm25,dspy_initial,0.0206,0.0084,0.0335,88,42,117
23,bm25,dspy_optimized,0.0151,-0.0046,0.0359,103,74,70
32,ocr,multiquery,-0.0094,-0.0186,-0.0009,43,53,151
41,ocr,dspy_initial,-0.0063,-0.0143,0.0003,34,33,180
50,ocr,dspy_optimized,-0.0099,-0.0194,-0.0007,47,66,134
59,image,multiquery,-0.0072,-0.0239,0.0100,69,77,101
68,image,dspy_initial,-0.0041,-0.0169,0.0090,61,72,114
77,image,dspy_optimized,-0.0101,-0.0246,0.0044,64,70,113
86,combined,multiquery,-0.0220,-0.0376,-0.0066,66,86,95


## 6. Does prompt optimization itself help?

Each main-table pipeline selects its own fusion parameters on validation. Its improvement can reflect both the query wording and those parameters. To isolate the instruction more closely, the table below compares initial and optimized DSPy queries using **the optimized pipeline's identical retrieval/fusion parameters** for both. The initial prompt still retains its independently tuned row in the main comparison.

The supplementary previous-setting controls also preserve BM25's former k1=1.5, b=0.75 and the former 50/50 OCR/image fusion, so changing those defaults is not confused with query expansion.

In [6]:
matched = pd.read_csv(RESULTS / 'matched_dspy_control.csv')
display(matched[['backend','family','queries','hit_at_10','recall_at_10','ndcg_at_10']].round(4))
controls = pd.read_csv(RESULTS / 'previous_setting_controls.csv')
test_ids = set(splits[(splits.split == 'test') & (splits.language == 'english')].query_id)
display(controls[(controls.query_id.isin(test_ids)) & (controls.k == 10)].groupby('method')[['hit','recall','ndcg']].mean().round(4))

,backend,family,queries,hit_at_10,recall_at_10,ndcg_at_10
0,bm25,dspy_initial,247,0.8907,0.5831,0.5257
1,bm25,dspy_optimized,247,0.8907,0.5823,0.5188
2,ocr,dspy_initial,247,0.8502,0.5633,0.5079
3,ocr,dspy_optimized,247,0.8502,0.5657,0.5066
4,image,dspy_initial,247,0.9150,0.6265,0.5789
5,image,dspy_optimized,247,0.9069,0.6179,0.5690
6,combined,dspy_initial,247,0.9069,0.6261,0.5766
7,combined,dspy_optimized,247,0.9150,0.6279,0.5712


,hit,recall,ndcg
method,,,
bm25_previous_default,0.8826,0.5720,0.5156
fusion_previous_50_50,0.9109,0.6191,0.5620


## 7. Full-dataset and subgroup results

All 1,854 variants were processed; none of the final evaluation rows are sample estimates. The main test conclusion uses English because that is the development objective. The full English and multilingual tables below are descriptive aggregates that also contain development questions. They must not be presented as independent held-out confirmations.

Language-specific changes show how querying an English corpus with translated questions behaves. Visual/table/text groupings derive from dataset content tags; they are not newly verified annotations.

In [7]:
all_en = summary[(summary.scope == 'language') & (summary['group'] == 'english') & (summary.k == 10)]
display(all_en.set_index('method')[['queries','hit','recall','ndcg']].sort_index().round(4))
language = summary[(summary.scope == 'split_language') & summary['group'].str.startswith('test/') & (summary.k == 10)].copy()
display(language.pivot(index='method', columns='group', values='ndcg').sort_index().round(4))
content = summary[(summary.scope == 'test_english_content') & (summary.k == 10)]
display(content.pivot(index='method', columns='group', values='ndcg').sort_index().round(4))

,queries,hit,recall,ndcg
method,,,,
bm25/dspy_initial,309,0.8932,0.5924,0.5398
bm25/dspy_optimized,309,0.9029,0.5978,0.5381
bm25/multiquery,309,0.8932,0.5983,0.5348
bm25/original,309,0.8900,0.5778,0.5221
combined/dspy_initial,309,0.9126,0.6397,0.5887
combined/dspy_optimized,309,0.9256,0.6449,0.5896
combined/multiquery,309,0.9159,0.6324,0.5727
combined/original,309,0.9223,0.6464,0.5894
image/dspy_initial,309,0.9094,0.6406,0.5905


group,test/english,test/french,test/german,test/italian,test/portuguese,test/spanish
method,,,,,,
bm25/dspy_initial,0.5243,0.0800,0.1927,0.1626,0.0947,0.0468
bm25/dspy_optimized,0.5188,0.0922,0.1973,0.1831,0.1044,0.0548
bm25/multiquery,0.5191,0.0845,0.1988,0.1735,0.0958,0.0421
bm25/original,0.5037,0.0702,0.1758,0.1492,0.0916,0.0445
combined/dspy_initial,0.5743,0.4937,0.4948,0.5048,0.4887,0.5202
combined/dspy_optimized,0.5712,0.4918,0.4800,0.4994,0.4932,0.5185
combined/multiquery,0.5540,0.4725,0.4747,0.4909,0.4645,0.4999
combined/original,0.5760,0.4964,0.4945,0.5055,0.4926,0.5242
image/dspy_initial,0.5750,0.4824,0.4804,0.5037,0.4909,0.5150


group,chart_or_other_visual,table,text_or_other
method,,,
bm25/dspy_initial,0.5264,0.4424,0.6215
bm25/dspy_optimized,0.5583,0.4438,0.6011
bm25/multiquery,0.5247,0.4271,0.6276
bm25/original,0.4766,0.4290,0.5977
combined/dspy_initial,0.6652,0.5372,0.6020
combined/dspy_optimized,0.6978,0.5327,0.5939
combined/multiquery,0.6686,0.5237,0.5692
combined/original,0.6568,0.5365,0.6083
image/dspy_initial,0.6991,0.5314,0.6043


## 8. Examples where expansion helped or hurt

These are the largest held-out English nDCG@10 gain and loss for MultiQuery within each backend, selected for inspection after evaluation. They explain behavior; they are not representative-sample estimates. Check whether the rewrite preserved dates, entities and the requested financial relationship, and whether fusion promoted relevant pages or lexical distractors.

In [8]:
examples = load('qualitative_examples.json')
display(pd.DataFrame(examples)[['backend','example_type','query_id','question','ndcg_at_10_change']].round(4))
example = next(x for x in examples if x['backend'] == 'image' and x['example_type'] == 'largest_loss')
print('Question:', example['question'])
print('Five generated alternatives:')
for i, query in enumerate(example['generated_queries'], 1):
    print(f'{i}. {query}')
print('Judged pages / grades:', example['judged_pages'])
print('Original top 10:', example['original_top10'])
print('Expanded top 10:', example['expanded_top10'])

,backend,example_type,query_id,question,ndcg_at_10_change
0,bm25,largest_gain,206,What types of assets were acquired in loan sat...,0.6845
1,bm25,largest_loss,286,What was the increase in the aggregate fair va...,-0.6438
2,ocr,largest_gain,11,How did the total impact of valuation adjustme...,0.3155
3,ocr,largest_loss,100,Morgan Stanley total Average Management VaR in...,-0.5000
4,image,largest_gain,104,By how much were Citigroup's Other Real Estate...,0.6309
5,image,largest_loss,100,Morgan Stanley total Average Management VaR in...,-0.5693
6,combined,largest_gain,234,"Could downward shocks, such as 200-bps cut, re...",0.5000
7,combined,largest_loss,227,What percent loss of firmwide total capital de...,-0.5694


Question: Morgan Stanley total Average Management VaR in 2024.
Five generated alternatives:
1. Morgan Stanley average management value at risk 2024.
2. Morgan Stanley total VaR for management 2024.
3. Morgan Stanley overall management risk assessment 2024.
4. Morgan Stanley average risk exposure 2024.
5. Morgan Stanley total management risk metrics 2024.
Judged pages / grades: {'2239': 2}
Original top 10: [2239, 2240, 53, 2237, 452, 1070, 1069, 2281, 2018, 2211]
Expanded top 10: [2237, 2240, 452, 2239, 2281, 2241, 2242, 2203, 2224, 2018]


## 9. Runtime, API usage and validation

GPT-4o-mini generates question alternatives; local Qwen3-VL encodes them on the GPU. The page embeddings are reused, so no new page-indexing time is claimed. Candidate search uses cached vectors on the CPU, followed by RRF.

API latency, batched GPU embedding time and cached lookup/fusion time are different measurements. The recorded lookup timings are cache/order dependent and exclude generation and encoding; they are not end-to-end query latency. The API cost estimate uses saved unique successful request usage at $0.15 per million input tokens and $0.60 per million output tokens, without cache discounts. It excludes any interrupted/unrecorded calls and is not a billing statement. Alternative queries are generated in sets of five even when a pipeline uses only one, so costs do not represent a separately optimized one-rewrite deployment.

Raw API caches and credentials stay outside the upload package. The export includes the generated queries, sanitized usage, frozen settings, per-query metrics and top-20 rankings. A separate calculation checks every saved metric row from the rankings and qrels.

In [9]:
usage = pd.read_csv(RESULTS / 'api_usage.csv')
batch_records = [batch for run in manifest['query_embedding_batches'] for batch in run]
display(pd.Series({
    'Saved unique successful API requests': manifest['api_requests'],
    'Recorded prompt tokens': manifest['prompt_tokens'],
    'Recorded output tokens': manifest['completion_tokens'],
    'Recorded-usage cost estimate (USD)': round(manifest['estimated_api_usd'], 4),
    'Median API request latency (seconds)': round(manifest['api_latency_median_seconds'], 3),
    'P95 API request latency (seconds)': round(manifest['api_latency_p95_seconds'], 3),
    'New query vectors encoded (includes validation-only prompts)': sum(b['count'] for b in batch_records),
    'GPU embedding batch seconds (excludes model loading)': round(sum(b['seconds'] for b in batch_records), 2),
    'Exported final query-generation records': manifest['exported_query_generation_records'],
}))
print('Format-repair counts:', manifest['format_repair_counts'])
display(pd.Series(load('independent_audit.json')))
display(pd.Series(manifest['versions']))

Saved unique successful API requests                            7.639000e+03
Recorded prompt tokens                                          1.141250e+06
Recorded output tokens                                          9.971090e+05
Recorded-usage cost estimate (USD)                              7.695000e-01
Median API request latency (seconds)                            2.046000e+00
P95 API request latency (seconds)                               3.274000e+00
New query vectors encoded (includes validation-only prompts)    3.596500e+04
GPU embedding batch seconds (excludes model loading)            2.977000e+02
Exported final query-generation records                         7.416000e+03
dtype: float64

Format-repair counts: {'dspy_initial': 0, 'dspy_seed_19': 0, 'mq_financial': 0, 'mq_general': 1}


metric_rows_independently_checked                                               118656
queries                                                                           1854
headline_summary_rows_checked                                                       64
test_english_queries                                                               247
methods                                                                             16
translation_group_split_overlap                                                      0
bootstrap_replicates                                                              2000
interval_note                        Descriptive pointwise paired intervals; no mul...
status                                                                          passed
dtype: object

dspy                       3.5.0
langchain-classic          1.0.8
langchain-core             1.8.2
optuna                     5.1.0
transformers              4.57.6
torch                2.7.0+cu128
numpy                      2.5.4
pyarrow                   26.0.0
requests                  2.34.2
dtype: object

## 10. Interpretation and next steps

On the 247 English test questions, BM25 nDCG@10 rises from **0.5037** with the original question to **0.5191** with MultiQuery and **0.5243** with the initial DSPy prompt. The optimized DSPy pipeline reaches **0.5188**, so optimization does not improve on the initial pipeline here.

For the dense backends, expansion does **not** improve test nDCG@10 over the original question. Original dense image retrieval records **0.5791**, versus **0.5719** for MultiQuery and **0.5690** for optimized DSPy. Yet image MultiQuery Hit@10 increases from **0.9069** to **0.9150**: finding some relevant page and ranking all relevant pages well are different outcomes. These figures use frozen development-selected settings.

Use the optimizer-held-out English table to decide whether expansion improves each retrieval backend. Compare optimized DSPy against both original questions and its initial prompt, including the matched-parameter control. More alternatives or a higher development score do not guarantee a test improvement.

This experiment tests retrieval of labeled pages. It does not establish answer completeness, factual correctness, visual reading accuracy, or gains from a larger context budget. A subsequent reader experiment can keep the reader and evidence budget fixed across the selected page rankings. Fine-grained evidence sufficiency still requires reviewed fact/span annotations and an explicit policy for alternative supporting sources.

Sources: [ViDoRe Finance–EN](https://huggingface.co/datasets/vidore/vidore_v3_finance_en), [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B), [LangChain MultiQueryRetriever](https://reference.langchain.com/python/langchain-classic/retrievers/multi_query/MultiQueryRetriever), [DSPy MIPROv2](https://dspy.ai/current/api/optimizers/MIPROv2/).